# burner_redis smoke test

Rust-backed async Redis client (built via maturin). No external Redis server is
available in the test environment, so cells exercise the import surface, type
discovery, and offline construction / error paths only.

In [ ]:
# Inspect the top-level public API surface of burner_redis.
import burner_redis

public_names = [n for n in dir(burner_redis) if not n.startswith('_')]
print(f"Public names ({len(public_names)}):")
for name in public_names:
    obj = getattr(burner_redis, name)
    print(f"  {name}: {type(obj).__name__}")

assert public_names, "burner_redis exposes no public names"

In [ ]:
# Locate the primary client class (if any) and inspect its methods.
import inspect

import burner_redis

client_candidates = [
    name for name in dir(burner_redis)
    if not name.startswith('_')
    and inspect.isclass(getattr(burner_redis, name, None))
]
print(f"Class-like top-level attrs: {client_candidates}")

for cls_name in client_candidates[:5]:
    cls = getattr(burner_redis, cls_name)
    methods = [m for m in dir(cls) if not m.startswith('_')]
    print(f"\n{cls_name} methods ({len(methods)}): {methods[:15]}")

In [ ]:
# Verify that constructing a client with an invalid URL does NOT segfault.
# We only want a clean Python-level exception (TypeError / ValueError / OSError /
# a burner_redis-defined error). Any of those is acceptable here.
import inspect

import burner_redis

client_classes = [
    getattr(burner_redis, name)
    for name in dir(burner_redis)
    if not name.startswith('_')
    and inspect.isclass(getattr(burner_redis, name, None))
]

# Exception classes that burner_redis itself exports count as clean, expected errors too.
package_errors = tuple(cls for cls in client_classes if issubclass(cls, BaseException))
acceptable_errors = (TypeError, ValueError, OSError) + package_errors

tested_any = False
for cls in client_classes:
    try:
        cls('not-a-valid-redis-url')
        print(f"{cls.__name__}('not-a-valid-redis-url') -> constructed (no error)")
    except TypeError as e:
        print(f"{cls.__name__}: TypeError (likely needs different args): {e}")
    except acceptable_errors as e:
        print(f"{cls.__name__}: {type(e).__name__} (a clean, acceptable error): {str(e)[:120]}")
    except Exception as e:
        print(f"{cls.__name__}: unexpected {type(e).__name__}: {str(e)[:120]}")
        raise
    tested_any = True

print(f"\nTested {len(client_classes)} class(es) without crashing the interpreter.")

In [ ]:
# Final sanity assertion: the package is importable, has a version (or at least
# a __name__), and the underlying native extension loaded without crashing.
import sys

import burner_redis

assert 'burner_redis' in sys.modules, "burner_redis not in sys.modules after import"
assert burner_redis.__name__ == 'burner_redis'

# A maturin-built package typically ships a compiled .so / .pyd inside its dir.
# We don't require it (a pure-python shim is allowed), but log what we see.
pkg_path = getattr(burner_redis, '__path__', None)
if pkg_path:
    import os
    entries = sorted(os.listdir(list(pkg_path)[0]))
    print(f"Package directory entries: {entries}")
else:
    print(f"burner_redis is a single-file module at: {burner_redis.__file__}")

print("\nburner_redis smoke test PASSED")
assert True